# 02 — Control de calidad, clustering y tipos celulares

**Taller de célula única CIAD**

Adaptado del *Guided Clustering Tutorial* de Seurat:
<https://satijalab.org/seurat/articles/pbmc3k_tutorial>

Mantén esa página abierta — todo aquí corresponde a ella, así que puedes volver al
original después del taller.

**Los datos.** 2,700 células mononucleares de sangre periférica (PBMCs) de un
donante sano, secuenciadas por 10x Genomics. Suficientemente pequeño para correr
en un taller, suficientemente variado para contener casi todos los tipos celulares
inmunes que uno querría encontrar.

**Qué hacemos.** Partir de una matriz de counts sin ninguna etiqueta, y terminar
con tipos celulares nombrados.

1. control de calidad — descartar las gotas que no son realmente células
2. normalización y selección de genes
3. PCA, y después clustering
4. UMAP, para ver el resultado
5. genes marcadores, y nombrar los clusters

**Cómo ejecutar.** *Shift + Enter* corre una celda. Ejecútalas en orden, de arriba
hacia abajo.

Si se desconecta el runtime, vuelve a correr la celda de preparación y luego la
celda de checkpoint de la sección en la que ibas — no tendrás que empezar de
nuevo.

## Preparación

Esto instala Seurat y todo lo demás en la máquina temporal que te dio Colab.
Alrededor de un minuto la primera vez, segundos si la vuelves a correr.

In [ ]:
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup.R")

## 1. Los datos

Descargamos los counts desde 10x Genomics. Nota que esto lo descarga la máquina de
Colab, no tu laptop — los 7 MB nunca pasan por el wifi del congreso.

Los archivos son la salida estándar de 10x: tres archivos que describen una matriz
dispersa.

- `matrix.mtx` — los counts
- `barcodes.tsv` — una línea por célula
- `genes.tsv` — una línea por gen

In [ ]:
url <- "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"

download.file(url, "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

list.files("filtered_gene_bc_matrices/hg19")

`Read10X()` lee esos tres archivos en una sola matriz dispersa: genes en
las filas, células en las columnas.

Lo disperso importa. La mayoría de las entradas son cero — un gen dado no se
detecta en una célula dada — y guardar solo los valores distintos de cero es lo
que permite que esto quepa en memoria.

In [ ]:
pbmc.data <- Read10X(data.dir = "filtered_gene_bc_matrices/hg19")

cat("genes:", nrow(pbmc.data), "\n")
cat("cells:", ncol(pbmc.data), "\n")

# a corner of the matrix: "." is a zero that is not stored
pbmc.data[c("CD3D", "TCL1A", "MS4A1"), 1:20]

### El objeto Seurat

`CreateSeuratObject()` envuelve la matriz junto con todo lo que estamos por
calcular — métricas de calidad, clusters, coordenadas UMAP — en un solo objeto.

Aquí se aplican dos filtros, deliberadamente laxos:

- `min.cells = 3` — descarta genes vistos en menos de 3 células. No aportan
  información y solo cuestan memoria.
- `min.features = 200` — descarta gotas con menos de 200 genes detectados. Casi
  nunca son células intactas.

In [ ]:
pbmc <- CreateSeuratObject(
  counts       = pbmc.data,
  project      = "pbmc3k",
  min.cells    = 3,
  min.features = 200
)

pbmc

## 2. Control de calidad

No podemos ver por un microscopio, así que juzgamos cada gota por sus counts. Tres
números hacen la mayor parte del trabajo:

| métrica | qué es | qué sugiere un valor extremo |
|---|---|---|
| `nFeature_RNA` | genes detectados en la célula | muy bajo: gota vacía o célula muriendo. muy alto: dos células en una gota |
| `nCount_RNA` | total de moléculas en la célula | lo mismo que arriba |
| `percent.mt` | % de counts de genes mitocondriales | alto: la membrana se rompió, el RNA citoplásmico se fugó, el mitocondrial se quedó |

Las dos primeras las calcula `CreateSeuratObject()`. La tercera la agregamos
nosotros: los símbolos de los genes mitocondriales humanos empiezan con `MT-`.

In [ ]:
pbmc[["percent.mt"]] <- PercentageFeatureSet(pbmc, pattern = "^MT-")

head(pbmc@meta.data, 5)

### ✏️ Ejercicio 1

Los genes de proteínas ribosomales son otra señal común de calidad. Sus símbolos
empiezan con `RPS` o `RPL`.

Agrega una columna `percent.ribo` al objeto, y luego mira su distribución.

Completa el espacio en blanco:

In [ ]:
# pbmc[["percent.ribo"]] <- PercentageFeatureSet(pbmc, pattern = ______)

summary(pbmc$percent.ribo)

### Mirando las distribuciones

Un violin plot por métrica. Buscamos dónde está el grueso de las células, y las
colas que quizá queramos recortar.

In [ ]:
VlnPlot(pbmc,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        ncol     = 3)

Las métricas son más fáciles de juzgar de dos en dos. Lee el panel
izquierdo como "¿las células con muchos counts tienen alto contenido
mitocondrial?", y el derecho como "¿las células con muchos counts tienen más
genes?" — el derecho debería ser una relación estrecha, y las células que se salen
de ella son sospechosas.

In [ ]:
p1 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "percent.mt")
p2 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")

p1 + p2

### Filtrado

Los umbrales del tutorial: más de 200 y menos de 2,500 genes, menos de 5% de
contenido mitocondrial.

Estos números **no son universales**. Salen de mirar los gráficos de arriba, para
este tejido y este protocolo. En otro conjunto de datos habría que volver a
mirar.

In [ ]:
cells_before <- ncol(pbmc)

pbmc <- subset(pbmc,
               subset = nFeature_RNA > 200 &
                        nFeature_RNA < 2500 &
                        percent.mt   < 5)

cat("before:", cells_before, "cells\n")
cat("after :", ncol(pbmc), "cells\n")
cat("removed:", cells_before - ncol(pbmc),
    sprintf("(%.1f%%)\n", 100 * (cells_before - ncol(pbmc)) / cells_before))

### ✏️ Ejercicio 2

¿Qué tan sensible es esa decisión?

Vuelve a aplicar el filtro sobre una copia del objeto con un corte mitocondrial
más **estricto** de 2.5%, y reporta cuántas células más pierdes. No sobrescribas
`pbmc`.

Completa los espacios en blanco:

In [ ]:
# strict <- subset(pbmc_unfiltered,
#                  subset = nFeature_RNA > 200 &
#                           nFeature_RNA < 2500 &
#                           percent.mt   < ______)
# ncol(strict)

# Hint: you no longer have the unfiltered object — pbmc was overwritten above.
# What is the cheapest way to get it back?

## 3. Normalización

Las células difieren en cuánto RNA se capturó, por razones puramente técnicas. Sin
corregirlo, la señal más fuerte en los datos es la profundidad de secuenciación y
no la biología.

`LogNormalize` divide cada count entre el total de la célula, multiplica por
10,000, y aplica `log1p`. El resultado va a una nueva layer; los counts crudos
quedan intactos.

In [ ]:
pbmc <- NormalizeData(pbmc,
                      normalization.method = "LogNormalize",
                      scale.factor         = 10000)

# raw counts and normalised values now live side by side
Layers(pbmc[["RNA"]])

## 4. Genes variables

La mayoría de los genes se expresan más o menos igual en todas las células. Meten
ruido y costo de cómputo sin ayudar a distinguir tipos celulares.

`FindVariableFeatures()` se queda con los 2,000 genes cuya varianza es más alta
*dada su media* — el método `vst` — ya que la varianza cruda simplemente
seleccionaría los genes muy expresados.

In [ ]:
pbmc <- FindVariableFeatures(pbmc, selection.method = "vst", nfeatures = 2000)

top10 <- head(VariableFeatures(pbmc), 10)
top10

In [ ]:
p <- VariableFeaturePlot(pbmc)
LabelPoints(plot = p, points = top10, repel = TRUE)

Mira lo que salió: `PPBP` (plaquetas), `LYZ` (monocitos), `GNLY` y
`NKG7` (células NK), `S100A8` (neutrófilos y monocitos). El método no sabe nada de
inmunología, y aun así los genes más variables son marcadores de los tipos
celulares presentes. Esa es toda la idea.

## 5. Escalado

El PCA se guía por la varianza, así que un gen muy expresado dominaría simplemente
por estar muy expresado. `ScaleData()` centra cada gen en cero y lo escala a
varianza uno, de modo que los genes se comparan en igualdad de condiciones.

Aquí escalamos todos los genes, lo que toma unos segundos. Por defecto solo se
escalan los genes variables — suficiente para el PCA, pero los heatmaps de más
adelante se ven mejor con todo escalado.

In [ ]:
pbmc <- ScaleData(pbmc, features = rownames(pbmc))

## 6. PCA

2,000 genes variables siguen siendo demasiadas dimensiones. El PCA las comprime en
unas cuantas decenas de componentes que capturan la estructura, y son esos
componentes — no los genes — los que usan el clustering y el UMAP.

In [ ]:
pbmc <- RunPCA(pbmc, features = VariableFeatures(pbmc), verbose = FALSE)

print(pbmc[["pca"]], dims = 1:5, nfeatures = 5)

Cada componente es una combinación ponderada de genes. Arriba están
impresos los cinco genes que más jalan de cada uno de los primeros cinco
componentes — y otra vez se leen como firmas de tipos celulares.

Un heatmap lo hace concreto. Las células están ordenadas por su puntaje en el
componente, los genes por su loading. Una estructura limpia de bloques significa
que el componente separa algo real.

In [ ]:
DimHeatmap(pbmc, dims = 1:6, cells = 500, balanced = TRUE)

### ¿Cuántos componentes?

Si tomas muy pocos pierdes estructura real. Si tomas demasiados haces clustering
sobre ruido.

El elbow plot muestra cuánta varianza explica cada componente. Donde la curva se
aplana, los componentes restantes son mayormente ruido.

In [ ]:
ElbowPlot(pbmc, ndims = 30)

### ✏️ Ejercicio 3

Mira el elbow plot y decide dónde se aplana.

El tutorial usa 10. ¿Es defendible según el gráfico? ¿Cambiaría mucho con 15?

Asigna a `n_dims` el número que defenderías, y lo usaremos en el resto del
notebook.

In [ ]:
n_dims <- ______   # replace with your choice, e.g. 10

cat("using", n_dims, "principal components\n")

## 7. Clustering

Dos pasos.

`FindNeighbors()` construye un grafo: cada célula se conecta con las células más
cercanas a ella en el espacio del PCA.

`FindClusters()` luego encuentra grupos de células más conectadas entre sí que con
el resto — comunidades en ese grafo.

`resolution` controla el nivel de detalle. Más alto da más clusters y más
pequeños. No hay un valor correcto; 0.4–1.2 es el rango habitual para unas pocas
miles de células.

In [ ]:
pbmc <- FindNeighbors(pbmc, dims = 1:n_dims)
pbmc <- FindClusters(pbmc, resolution = 0.5)

table(Idents(pbmc))

## 8. UMAP

UMAP coloca cada célula en un mapa 2D, tratando de mantener como vecinas a las
células que eran vecinas en el espacio del PCA.

Dos advertencias, que vale la pena decir en voz alta antes de que alguien lea de
más en una figura:

- **La distancia entre clusters significa poco.** Dos clusters lejanos en un UMAP
  no son necesariamente más distintos que dos cercanos.
- **El UMAP no define los clusters.** Los clusters se calcularon en el espacio del
  PCA, en `n_dims` dimensiones. El UMAP solo los dibuja.

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:n_dims, verbose = FALSE)

DimPlot(pbmc, reduction = "umap", label = TRUE) + NoLegend()

### 💾 Checkpoint

Si te quedaste atrás, o se cayó el runtime, este es el punto que vale la pena
guardar.

La celda de abajo escribe el objeto en el disco de la máquina de Colab.
Desaparece cuando el runtime se apaga, pero sobrevive a que vuelvas a correr
celdas por accidente.

In [ ]:
saveRDS(pbmc, "pbmc_clustered.rds")

# To come back to this point later:
# pbmc <- readRDS("pbmc_clustered.rds")

cat("saved:", round(file.size("pbmc_clustered.rds") / 1e6, 1), "MB\n")

## 9. Genes marcadores

Tenemos clusters con números. Para nombrarlos necesitamos saber qué expresa cada
uno que los demás no.

`FindAllMarkers()` compara cada cluster contra todas las células restantes, un
cluster a la vez. `only.pos = TRUE` conserva solo los genes que están *más altos*
en el cluster, que es lo que quieres para ponerles nombre.

Es la celda más lenta del notebook — alrededor de un minuto.

In [ ]:
markers <- FindAllMarkers(pbmc, only.pos = TRUE, verbose = FALSE)

markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 3) %>%
  ungroup() %>%
  as.data.frame()

Dos formas de mirar un marcador: dónde se expresa en el mapa, y qué tan
fuerte se expresa por cluster.

In [ ]:
FeaturePlot(pbmc,
            features = c("MS4A1", "CD3E", "CD14", "FCGR3A",
                         "GNLY", "LYZ", "FCER1A", "PPBP"),
            ncol     = 4)

In [ ]:
VlnPlot(pbmc, features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 2)

Un heatmap de los principales marcadores por cluster da el panorama
completo de una vez. Cada columna es una célula, agrupadas por cluster; cada fila
un gen.

In [ ]:
top_markers <- markers %>%
  group_by(cluster) %>%
  dplyr::filter(avg_log2FC > 1) %>%
  slice_head(n = 10) %>%
  ungroup()

DoHeatmap(pbmc, features = top_markers$gene) + NoLegend()

## 10. Nombrar los clusters

Este es el paso que ningún algoritmo hace por ti. Se trata de relacionar genes
marcadores con biología conocida.

Para PBMCs los marcadores canónicos están bien establecidos:

| marcadores | tipo celular |
|---|---|
| `IL7R`, `CCR7` | T CD4 naive |
| `IL7R`, `S100A4` | T CD4 de memoria |
| `CD14`, `LYZ` | monocitos CD14+ |
| `MS4A1` | B |
| `CD8A` | T CD8 |
| `FCGR3A`, `MS4A7` | monocitos FCGR3A+ |
| `GNLY`, `NKG7` | NK |
| `FCER1A`, `CST3` | células dendríticas |
| `PPBP` | plaquetas |

### ✏️ Ejercicio 4

Las etiquetas de abajo son las del tutorial, en el orden de clusters del tutorial.
**Tu numeración de clusters puede ser distinta** — depende del número de
componentes y de la resolución que elegiste.

Verifícalas contra tus propios marcadores antes de aceptarlas. Si tu cluster 3 no
es el de células B, el vector está mal para ti y hay que reordenarlo.

In [ ]:
# The order must match levels(pbmc) — check first:
levels(pbmc)

In [ ]:
new_ids <- c("Naive CD4 T", "CD14+ Mono", "Memory CD4 T", "B",
             "CD8 T", "FCGR3A+ Mono", "NK", "DC", "Platelet")

# only works if you have exactly as many clusters as labels
stopifnot(length(new_ids) == length(levels(pbmc)))

names(new_ids) <- levels(pbmc)
pbmc <- RenameIdents(pbmc, new_ids)

DimPlot(pbmc, reduction = "umap", label = TRUE, pt.size = 0.5) + NoLegend()

Guarda las etiquetas en algún lugar permanente. `Idents()` se
sobrescribe fácilmente por accidente; una columna de metadata no.

In [ ]:
pbmc$cell_type <- Idents(pbmc)

table(pbmc$cell_type)

In [ ]:
saveRDS(pbmc, "pbmc_annotated.rds")

cat("saved:", round(file.size("pbmc_annotated.rds") / 1e6, 1), "MB\n")

## Qué hicimos

De una matriz de counts sin etiquetas, a tipos celulares inmunes nombrados, en
unos diez pasos.

Vale la pena llevarse esto:

- **Los umbrales de calidad son criterio, no valores por defecto.** Miramos
  distribuciones y luego elegimos. Otro tejido, otros números.
- **El clustering ocurre en el espacio del PCA, no en el UMAP.** La figura es una
  proyección del resultado, no el resultado.
- **La resolución y la dimensionalidad son decisiones.** Cambian cuántos clusters
  obtienes. Si una conclusión solo se sostiene con una resolución, no es una
  conclusión.
- **Nombrar los clusters es la biología.** Todo lo anterior es contabilidad.

Siguiente notebook: qué pasa cuando los datos vienen de más de una muestra, y el
batch effect es más grande que la biología.

---

### Respuestas

<details>
<summary>Haz clic para desplegar</summary>

**Ejercicio 1**

```r
pbmc[["percent.ribo"]] <- PercentageFeatureSet(pbmc, pattern = "^RP[SL]")
```

**Ejercicio 2**

`pbmc` fue sobrescrito por el paso de filtrado, así que el objeto sin filtrar ya
no existe. La forma más barata de recuperarlo es reconstruirlo desde `pbmc.data`,
que sigue en memoria:

```r
unf <- CreateSeuratObject(pbmc.data, min.cells = 3, min.features = 200)
unf[["percent.mt"]] <- PercentageFeatureSet(unf, pattern = "^MT-")
strict <- subset(unf, subset = nFeature_RNA > 200 &
                               nFeature_RNA < 2500 &
                               percent.mt   < 2.5)
ncol(strict)
```

La lección de fondo: asignar un objeto filtrado sobre su propio nombre tira a la
basura justo lo que necesitas para revisar el filtro. Usa un nombre nuevo.

**Ejercicio 3**

Cualquier valor entre 10 y 15 es defendible. La curva está claramente plana
después de ~15, y la diferencia entre 10 y 15 es pequeña — que es justo el punto:
si tus clusters cambian por completo entre 10 y 15 componentes, nunca fueron
estables.

**Ejercicio 4**

Compara los principales marcadores de cada cluster con la tabla de arriba. Con 10
componentes y resolución 0.5 normalmente salen 9 clusters en el orden del
tutorial, pero no está garantizado. Si `stopifnot()` falla, tienes un número
distinto de clusters — baja la resolución, o escribe un vector de etiquetas que
corresponda a lo que realmente tienes.

</details>